In [1]:
# importer les modules pour la convolution
from keras.layers import Conv2D, MaxPooling2D
from tensorflow.keras.layers import Dense, Flatten, Dropout
from keras.models import Sequential

In [ ]:
classifier = Sequential()
# j'ajoute une couche de convulation de 32 filtres, avec une taille 3*3, la phoo entre avec 64*64*3 3 parce qu'elle est en couleur et la fonction d'activation est relu
classifier.add(Conv2D(32, (3, 3), input_shape=(64, 64, 3), activation='relu'))
# ajouter la couche de pooling, max pooling 2*2
classifier.add(MaxPooling2D(pool_size=(2, 2)))
# j'ajoute la couche flatten
classifier.add(Flatten())
# FULL CONNECTION
classifier.add(Dense(units=128, activation='relu'))
# sortie
classifier.add(Dense(units=1, activation='relu'))

In [ ]:
# je compile
classifier.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])


In [ ]:
# Artificial Image generator
from keras.preprocessing.image import ImageDataGenerator
# Data generator
train_datagen = ImageDataGenerator(
        rescale=1./255, # standardisation
        rotation_range=40, # tourner la photo
        width_shift_range=0.2, # zoomer la photo
        height_shift_range=0.2,
        shear_range=0.2,
        zoom_range=0.2,
        horizontal_flip=True)
test_datagen = ImageDataGenerator(rescale=1./255)

In [ ]:
# loading the training set
training_set = train_datagen.flow_from_directory('dataset/training_set',
                                                 target_size=(64, 64), # j'impose que la photo arrive en taille de 64,64
                                                 batch_size=32, # les photos vont venir en lot de 32
                                                 class_mode='binary') # c'est un probleme de classification binaire
testin_set = test_datagen.flow_from_directory('dataset/test_set',
                                            target_size=(64, 64),
                                            batch_size=32,
                                            class_mode='binary')

In [ ]:
# entrainement du modèle

model = classifier.fit(training_set,
                         steps_per_epoch=8000,
                         epochs=25,
                         validation_data=testin_set,
                         validation_steps=2000)

In [ ]:
# evaluation du training
import matplotlib.pyplot as plt
plt.plot(model.history['accuracy'])
plt.plot(model.history['val_accuracy'])
plt.title('model accuracy')
plt.ylabel('accuracy')
plt.xlabel('epoch')
plt.legend(['training Accuray', 'validation accuracy'], loc='upper left')

In [ ]:
# evaluation du loss
import matplotlib.pyplot as plt
plt.plot(model.history['loss'])
plt.plot(model.history['val_loss'])
plt.title('model loss')
plt.ylabel('loss')
plt.xlabel('epoch')
plt.legend(['training loss', 'validation loss'], loc='upper left')

In [ ]:
# prediction sur une nouvelle photo
import numpy as np
from keras.preprocessing import image
test_image = image.load_img('dataset/single_prediction/cat_or_dog_1.jpg', target_size = (64, 64))
test_image = image.img_to_array(test_image) # je transforme l'image en array
test_image = np.expand_dims(test_image, axis = 0) # j'ajoute une dimension avec expand_dims
result = classifier.predict(test_image)

In [ ]:
# chercher les indices
training_set.class_indices

In [ ]:
for key, value in training_set.class_indices.items():
  if (result == value):
    print(key)

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
